In [1]:
import os
import pandas as pd
from google import genai
from google.genai import types
from PIL import Image
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer, util
from scipy.stats import spearmanr



Description function

In [ ]:
client = genai.Client(api_key="ENTER API KEY HERE") # Replace with your actual API key

# We use Flash-Lite for maximum value/speed
model_id = "gemini-2.5-flash-lite" 

def generate_description(image_path, prompt, model_id):
    sketch = Image.open(image_path)

    response = client.models.generate_content(
        model=model_id,
        contents=[prompt, sketch]
    )
    
    return response.text

def get_folder(category, condition):
    
    # get folder 
    if condition == "_control":
        data_folder = f"../../image_similarities/drawings_draw3D/used_drawings_exp1/control/{category}/"
    elif condition == "":
        data_folder = f"../../image_similarities/drawings_draw3D/used_drawings_exp1/own/{category}/"
    elif condition == "_photos":
        data_folder = f"../../image_similarities/pictures/{category}/"
    else:
        raise ValueError(f"Unsupported condition: {condition}")
      
    print(f"Selected folder: {data_folder}")  
    return data_folder


# 3. Load Images and Captions
def get_job_list(category, condition):

    # get folder
    print("Get images and captions")
    folder = get_folder(category, condition)

    # Filter for category files (e.g., 'kit' in filename)
    if condition == "_control":
        exts = (".jpg")
        files = [f for f in os.listdir(folder) if f.lower().endswith(exts) and category[0:3] in f
                 and not "'" in f and not "gen" in f and "copy" in f] 
    elif condition == "":
        exts = (".jpg")
        files = [f for f in os.listdir(folder) if f.lower().endswith(exts) and category[0:3] in f 
                 and not "'" in f and not "gen" in f and not "copy" in f] 
    elif condition == "_photos":
        exts = (".png")
        files = [f for f in os.listdir(folder) if f.lower().endswith(exts) and category[0:3] in f] 

    # Load captions
    captions = pd.DataFrame() # empty dataframe for photos condition  
    if condition == "_control" or condition == "":
        caption_file = os.path.join(folder, f"{category}{condition}_objects.csv")
        captions = pd.read_csv(caption_file, sep=",", header=0)
        captions = captions.iloc[:, 1:] # remove first column

    return sorted(files), captions, folder

def get_description(cate, condition, max_retries=5, retry_delay=10, image_repetitions=1, stimuli_subset=None):    
    
    print(f"Getting descriptions for {condition} on {cate}")
    
    # get files and folder
    files, captions, data_folder = get_job_list(cate, condition)

    if stimuli_subset is not None:
        files = files[int(stimuli_subset[0]):int(stimuli_subset[1])]

    if image_repetitions > 1:   
        files = files*image_repetitions
        files.sort()   

        if captions.shape[0] > 5:
            captions = captions.loc[captions.index.repeat(image_repetitions)].reset_index(drop=True)
           
    print(len(files))
    print(captions.shape)

    csv_log_file = os.path.join(data_folder, f"gem_2-5_descriptions_{cate}{condition}.csv")

    print(f"🚀 Starting batch. Data will be saved to: {csv_log_file}")

    # 1. INITIALIZE FILE: Use 'w' mode to create/clear the file and write the header
    with open(csv_log_file, "w", encoding="utf-8") as f_init:
        f_init.write("filename,original_caption,detailed_description\n")

    # 2. BATCH LOOP: open in 'a' mode for safe appending during the loop
    with open(csv_log_file, "a", encoding="utf-8") as f_log:
        for i, filename in enumerate(files):
            input_path = os.path.join(data_folder, filename)
            
            if condition == "_control" or condition == "":
                objects = captions.iloc[i].astype(str)
                objects = objects[objects != 'nan']
                current_desc = ", ".join(objects)

                prompt = f"""
                Analyze this line drawing of a {cate} including {current_desc}. 
                Provide a detailed description of approximately 200 words in natural language.
                Include all visible objects, furniture, and specific layout details.
                Focus on the content and layout of the drawing, not the artistic style or quality.
                Format as a single clean paragraph of text.
                """
            elif condition == "_photos":
                current_desc = "N/A for photos"
                prompt = f"""
                Analyze this photo of a {cate}. 
                Provide a detailed description of approximately 200 words in natural language.
                Include all visible objects, furniture, and specific layout details.
                Focus on the content and layout of the scene, not the photographic style or quality.
                Format as a single clean paragraph of text.
                """

            print(f"[{i+1}/{len(files)}] Processing: {filename}...", end=" ")

            success = False
            for attempt in range(max_retries):
                try:
                    detailed_brief = generate_description(input_path, prompt, model_id)
                    
                    # Sanitize text for CSV: remove newlines and escape quotes
                    clean_brief = detailed_brief.replace('\n', ' ').replace('"', '""')
                    clean_caption = current_desc.replace('"', '""')
                    
                    # Write formatted CSV row
                    f_log.write(f'"{filename}","{clean_caption}","{clean_brief}"\n')
                    f_log.flush() # Ensure it writes to disk immediately
                    
                    print("✅ Done.")
                    success = True
                    break 
                    
                except Exception as e:
                    # Exponential backoff: 10s, 40s, 90s...
                    wait_time = retry_delay * ((attempt + 1) ** 2)
                    print(f"⚠️ Attempt {attempt+1} failed: {e}. Retrying in {wait_time}s...")
                    time.sleep(wait_time)

            if not success:
                print(f"❌ Failed after {max_retries} attempts.")
                f_log.write(f'"{filename}","{current_desc}","ERROR: Generation Failed"\n')
                f_log.flush()

            time.sleep(2) # Tier 1 polite delay

Get descriptions from images and prompts

In [5]:
# 1. Setup paths and model
cates = ["kitchen", "bathroom"]
conditions = ["_control", ]

for cate in cates:
    for condition in conditions:  
        get_description(cate, condition,
                        max_retries=5, retry_delay=10,
                        image_repetitions=3, stimuli_subset=None
                        )
    



Getting descriptions for _control on kitchen
Get images and captions
Selected folder: ../../image_similarities/drawings_draw3D/used_drawings_exp1/control/kitchen/
105
(105, 23)
🚀 Starting batch. Data will be saved to: ../../image_similarities/drawings_draw3D/used_drawings_exp1/control/kitchen/gem_2-5_descriptions_kitchen_control.csv
[1/105] Processing: 101_kit_copy.jpg... ✅ Done.
[2/105] Processing: 101_kit_copy.jpg... ✅ Done.
[3/105] Processing: 101_kit_copy.jpg... ✅ Done.
[4/105] Processing: 102_kit_copy.jpg... ✅ Done.
[5/105] Processing: 102_kit_copy.jpg... ✅ Done.
[6/105] Processing: 102_kit_copy.jpg... ✅ Done.
[7/105] Processing: 103_kit_copy.jpg... ✅ Done.
[8/105] Processing: 103_kit_copy.jpg... ✅ Done.
[9/105] Processing: 103_kit_copy.jpg... ✅ Done.
[10/105] Processing: 104_kit_copy.jpg... ✅ Done.
[11/105] Processing: 104_kit_copy.jpg... ✅ Done.
[12/105] Processing: 104_kit_copy.jpg... ✅ Done.
[13/105] Processing: 105_kit_copy.jpg... ✅ Done.
[14/105] Processing: 105_kit_copy.jpg

Get MPNet Model 

In [3]:
from huggingface_hub import snapshot_download

# This downloads the files directly without checking for chat templates
model_path = snapshot_download(
    repo_id="sentence-transformers/all-mpnet-base-v2",
    revision="main",
    ignore_patterns=["additional_chat_templates"] 
)

print(f"✅ Model files are safely located at: {model_path}")

model = SentenceTransformer(model_path)

Fetching 28 files:   0%|          | 0/28 [00:00<?, ?it/s]

✅ Model files are safely located at: C:\Users\JLU-SU\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2\snapshots\e8c3b32edf5434bc2275fc9bab85f82640a19130


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: C:\Users\JLU-SU\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2\snapshots\e8c3b32edf5434bc2275fc9bab85f82640a19130
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Similarities of MPNet embeddings

In [6]:
def process_similarities_MPNet(csv_name, cate):

    if "photos" in csv_name:
        csv_path = os.path.join("../../image_similarities/pictures", cate, csv_name)
    elif "_control" in csv_name:
        csv_path = os.path.join("../../image_similarities/drawings_draw3D/used_drawings_exp1/control", cate, csv_name)
    else:
        csv_path = os.path.join("../../image_similarities/drawings_draw3D/used_drawings_exp1/own", cate, csv_name)

    if not os.path.exists(csv_path):
        print(f"❌ CSV file not found: {csv_path}")
        return
    
    df = pd.read_csv(csv_path)

    # If the expected columns are missing, create them by concatenating all columns into a single description
    if 'detailed_description' not in df.columns or 'filename' not in df.columns:
        df['detailed_description'] = df.apply(lambda row: ', '.join([str(row[col]) for col in df.columns]), axis=1)
        df['filename'] = range(len(df))
        df = df[['filename', 'detailed_description']] 

    descriptions = df['detailed_description'].fillna("").tolist()
    filenames = df['filename'].tolist()

    print(f"🔍 Encoding {len(descriptions)} descriptions...")
    
    # 1. Generate Embeddings
    embeddings = model.encode(descriptions, convert_to_tensor=True)
    
    # 2. Calculate Dissimilarity
    spearman_corr, _ = spearmanr(embeddings, axis=1)
    spearman_dissimilarity = 1 - spearman_corr
    
    # 3. Create DataFrame
    sim_df = pd.DataFrame(spearman_dissimilarity, index=filenames, columns=filenames)
    
    # 4. Save and Plot
    base_name = os.path.splitext(csv_name)[0]
    if "photos" in csv_name:
        cos_out = os.path.join("../../image_similarities/pictures", cate, f"{base_name}_MPNet_spearman.csv")
    elif "_control" in csv_name:
        cos_out = os.path.join("../../image_similarities/drawings_draw3D/used_drawings_exp1/control", cate, f"{base_name}_MPNet_spearman.csv")
    else:
        cos_out = os.path.join("../../image_similarities/drawings_draw3D/used_drawings_exp1/own", cate, f"{base_name}_MPNet_spearman.csv")
    sim_df.to_csv(cos_out)

    print(f"Spearman dissimilarity matrix saved to: {cos_out} ✅")
    
    #plt.figure(figsize=(10, 8))
    #sns.heatmap(sim_df, cmap='viridis', vmin=0.3, vmax=1.0)
    #plt.title(f"Similarity Matrix: {csv_name}")
    #plt.show()
    
    return sim_df

# Execute for gemini-2.5 descriptions
kit_matrix_desc_mpnet = process_similarities_MPNet("gem_2-5_descriptions_bathroom.csv", 'bathroom')
bat_matrix_desc_mpnet = process_similarities_MPNet("gem_2-5_descriptions_kitchen.csv", 'kitchen')

kit_ctr_matrix_desc_mpnet = process_similarities_MPNet("gem_2-5_descriptions_bathroom_control.csv", 'bathroom')
bat_ctr_matrix_desc_mpnet = process_similarities_MPNet("gem_2-5_descriptions_kitchen_control.csv", 'kitchen')

kit_photo_matrix_desc_mpnet = process_similarities_MPNet("gem_2-5_descriptions_bathroom_photos.csv", 'bathroom')
bat_photo_matrix_desc_mpnet = process_similarities_MPNet("gem_2-5_descriptions_kitchen_photos.csv", 'kitchen')
    


🔍 Encoding 105 descriptions...
Spearman dissimilarity matrix saved to: ../../image_similarities/drawings_draw3D/used_drawings_exp1/own\bathroom\gem_2-5_descriptions_bathroom_MPNet_spearman.csv ✅
🔍 Encoding 105 descriptions...
Spearman dissimilarity matrix saved to: ../../image_similarities/drawings_draw3D/used_drawings_exp1/own\kitchen\gem_2-5_descriptions_kitchen_MPNet_spearman.csv ✅
🔍 Encoding 105 descriptions...
Spearman dissimilarity matrix saved to: ../../image_similarities/drawings_draw3D/used_drawings_exp1/control\bathroom\gem_2-5_descriptions_bathroom_control_MPNet_spearman.csv ✅
🔍 Encoding 105 descriptions...
Spearman dissimilarity matrix saved to: ../../image_similarities/drawings_draw3D/used_drawings_exp1/control\kitchen\gem_2-5_descriptions_kitchen_control_MPNet_spearman.csv ✅
🔍 Encoding 144 descriptions...
Spearman dissimilarity matrix saved to: ../../image_similarities/pictures\bathroom\gem_2-5_descriptions_bathroom_photos_MPNet_spearman.csv ✅
🔍 Encoding 111 descriptions.

In [9]:
bat_ctr_matrix_desc_mpnet = process_similarities_MPNet("gem_2-5_descriptions_kitchen_control.csv", 'kitchen')

🔍 Encoding 35 descriptions...
Cosine dissimilarity matrix saved to: ../../image_similarities/drawings_draw3D/used_drawings_exp1/control\kitchen\gem_2-5_descriptions_kitchen_control_MPNet_cosine.csv ✅
